In [1]:
!pip install -r requirements.txt

In [33]:
# Parameters
num_readers = 10
novelty_weight = 0.3
relevance_weight = 0.3
expectation_rubric = [
    "How similar do you find this story to the stories in the books you have read?",
    "Based on all the books you have read before, to what extent can you predict the plot progression in this story?",
    "Based on all the books you have read before, how steoretypical do you find the characters in the story?",
    "Based on all the books you have read before, how expected do you find the ending of the story?"
]
relevance_rubric = [
    "how much do you enjoy this book you just read based on your personal life experience?", 
    "how much do you resonate with this book you just read based on your personal life experience?"
]
quality_rubric = [
    "how cohenrent do you think the plot is?",
    "how consistent do you think the characters are?"
]
open_ended_feedback = [
    "based on your background and personal experience, what do you like about this story?",
    "based on your background and personal experience, what do you dislike about this story?"
]

In [15]:
# Prompts 
reading_decision_prompt = """
You see a fiction titled {title}, with the following summary: {summary}.\n Would you be interested in reading this book?
"""


In [50]:
import os
from genagents.genagents import GenerativeAgent
agent_bank_path = "agent_bank/populations/gss_agents/"
agents = []
for f in os.listdir(agent_bank_path):
    if len(agents) >= num_readers:
        break
    if os.path.isdir(os.path.join(agent_bank_path, f)) :
        try:
            agent = GenerativeAgent(agent_folder=os.path.join(agent_bank_path, f))
            agents.append(agent)
            print(f"The following agent has been added as a reader: {agent.scratch}")
        except:
            print(f"Not a valid agent path: {f}")

The following agent has been added as a reader: {'first_name': 'Eric', 'last_name': 'Miller', 'age': 31, 'sex': 'Male', 'ethnicity': 'Germany', 'race': 'White', 'detailed_race': 'White', 'hispanic_origin': 'Not Hispanic', 'street_address': '1234 Summit Avenue', 'city': 'Des Moines', 'state': 'IA', 'political_views': 'Moderate, middle of the road', 'party_identification': 'Independent (neither, no response)', 'residence_at_16': 'West North Central', 'same_residence_since_16': 'Same state, different city', 'family_structure_at_16': 'Lived with parents', 'family_income_at_16': 'Far above average', 'fathers_highest_degree': 'Graduate', 'mothers_highest_degree': 'High school', 'mothers_work_history': 'Yes', 'marital_status': 'Married', 'work_status': 'Working full time', 'military_service_duration': 'No active duty', 'religion': 'Protestant', 'religion_at_16': 'Protestant', 'born_in_us': 'Yes', 'us_citizenship_status': 'A U.S. citizen', 'highest_degree_received': 'Associate/junior college',

In [53]:
def get_reader_feedback(agents: list[GenerativeAgent], title: str, full_story: str, short_summary: str, timestep, price = 0.0):
    all_individual_feedback = []
    for agent in agents:
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} is looking at the book {title}...")
        # First, check if the story is worth reading at all
        question = {reading_decision_prompt.format(title = title, summary = short_summary): ['yes', 'no']}
        response = agent.categorical_resp(question)
        print(f"reasoning: {response['reasonings'][0]}")
        if response['responses'][0] == 'no':
            print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} decided to NOT to read {title}.")
            agent.remember(f"Came across a book titled {title} with the following summary: {short_summary}. Decided to not read it.", time_step=timestep)
            continue
        else:
            print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} decided to read {title}.")
        agent.remember(f"Came across a book titled {title} with the following summary: {short_summary}. Decided to read it. The book tells the following story: {full_story}", time_step=timestep)
        
        # Ask about relevance questions
        relevance_questions = {q: [1, 10] for q in relevance_rubric}
        responses = agent.numerical_resp(relevance_questions)
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} has the following thoughts in terms of the relevance of the book {title}.")
        print(responses)
        relevance_score = sum(responses["responses"]) / len(relevance_questions) / 10.0
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]}'s total relevance score on book {title}: {relevance_score}")
        
        # Ask about novelty questions
        expectation_questions = {q: [1, 10] for q in expectation_rubric}
        responses = agent.numerical_resp(expectation_questions)
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} has the following thoughts in terms of the novelty of the book {title}.")
        print(responses)
        expectation_score = sum(responses["responses"]) / len(expectation_questions) / 10.0
        novelty_score = 1.0 - expectation_score
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]}'s total novelty score on book {title}: {novelty_score}")

        # Ask about quality questions
        qualty_questions = {q: [1, 10] for q in quality_rubric}
        responses = agent.numerical_resp(qualty_questions)
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} has the following thoughts in terms of the quality of the book {title}.")
        print(responses)
        quality_score = sum(responses["responses"]) / len(qualty_questions) / 10.0
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]}'s total quality score on book {title}: {quality_score}")

        # Ask for open-ended feedback
        open_ended_response = []
        for question in open_ended_feedback:
            dialogue = [
                ("Interviewer", question),
            ]
            
            response = agent.utterance(dialogue)
            open_ended_response.append(response)
        print(f"{agent.scratch["first_name"]} {agent.scratch["last_name"]} has the following open-ended feedback on book {title}: {'\n'.join(open_ended_response)}")
        individual_feedback = {
            'reader_agent': agent,
            'total_score' : (quality_score + novelty_score + relevance_score) / 3.0,
            'novelty' : novelty_score,
            'relevance' : relevance_score,
            'quality' : quality_score,
            'qualitative_feedback': '\n'.join(open_ended_response)}
        
        all_individual_feedback.append(individual_feedback)
        print('-------------')
    aggregated_feedback = {
        'sold_percentage': len(all_individual_feedback) / len(agents),
        'aggregated_total_score': sum([f['total_score'] for f in all_individual_feedback]) / len(all_individual_feedback),
        'aggregated_novelty_score': sum([f['novelty'] for f in all_individual_feedback]) / len(all_individual_feedback),
        'aggregated_relevance_score': sum([f['relevance'] for f in all_individual_feedback]) / len(all_individual_feedback),
        'aggregated_quality_score': sum([f['quality'] for f in all_individual_feedback]) / len(all_individual_feedback),
        'aggregated_qualitative_feedback': "",
        'raw_feedback': all_individual_feedback
    }

    return aggregated_feedback


In [54]:
# Testing
title = "The Awakening of Chad Everly: A Hero’s Journey Through the Algorithm"
story_summary = """
Chad Everly, a 25-year-old Brooklyn designer absorbed in social media and existential drift, is jolted into action when a push notification announces that his carbon footprint is “trending.” Seeking meaning, he turns to a podcaster whose pseudo-spiritual advice he mistakes for mentorship and soon plunges into the absurdities of online culture, from discourse swamps to failed influencers. Hoping to assert his significance, he launches a Substack newsletter that garners almost no real readers, then joins a desert retreat run by a startup promising to fuse blockchain and empathy, where he bonds with an AI ethics researcher named Zara. When the startup inevitably collapses, Chad loses his savings and sense of self, forcing him into a period of reflection through gardening, reading, and genuine introspection. His renewed clarity leads him to write a viral insight about the futility of optimizing life, which ironically launches him into influencer fame. In the end, he returns to the digital world as a mindful influencer, fully aware that he and those around him are still caught in an inescapable loop of curated authenticity and algorithmic performance.
"""
full_story = """
- Chad’s mundane life: 25-year-old freelance designer in Brooklyn, obsessed with social media, oat-milk lattes, and existential dread.
- Inciting event: Push notification about his carbon footprint “trending” sparks a desire to make a difference.
- Mentor appears: Listens to podcaster Eliot Vox, who gives performative, pseudo-spiritual guidance.
- Digital trials: Faces the absurdities of online culture — the Discourse Swamp, cancelled influencers, and the Tower of the Take Economy.
- Newsletter attempt: Launches Substack “Thoughts, Probably”, gains minimal real readership.
- Tech-utopia retreat: Joins a desert startup retreat blending blockchain and empathy; meets AI ethics researcher Zara.
- Romantic/spiritual bonding: Discusses crypto-spiritual philosophy, feeling enlightened and connected.
- Collapse and crisis: Startup fails, Chad loses savings and digital identity, hits existential rock bottom.
- Rebirth and self-discovery: Rediscovers authenticity through gardening, reading, and introspection.
- Viral realization: Writes a viral insight about life and optimization, ironically gaining influencer fame.
- Return to the digital world: Becomes a mindful influencer, surrounded by others chasing curated authenticity, fully aware of the algorithmic loop he can’t escape.
"""

get_reader_feedback(agents, title, full_story, story_summary, 1)

Eric Miller is looking at the book The Awakening of Chad Everly: A Hero’s Journey Through the Algorithm...
reasoning: Considering Eric's age and background, he is likely to be intrigued by the contemporary issues presented in the book. His moderate views suggest an openness to exploring varied perspectives, which aligns with the book's themes of self-reflection and critique of modern society.
Eric Miller decided to read The Awakening of Chad Everly: A Hero’s Journey Through the Algorithm.
Eric Miller has the following thoughts in terms of the relevance of the book The Awakening of Chad Everly: A Hero’s Journey Through the Algorithm.
{'responses': [7, 8], 'reasonings': ['Eric seems to appreciate the themes of self-discovery and the absurdities of modern life, as reflected in his own situation and moderate political views. His life experiences align with the exploration of identity and purpose that Chad undergoes in the book.', "Given Eric's background of existential reflections and mode

{'sold_percentage': 0.8,
 'aggregated_total_score': 0.5614583333333333,
 'aggregated_novelty_score': 0.34687500000000004,
 'aggregated_relevance_score': 0.65,
 'aggregated_quality_score': 0.6875,
 'aggregated_qualitative_feedback': '',
 'raw_feedback': [{'reader_agent': <genagents.genagents.GenerativeAgent at 0x30475c860>,
   'total_score': 0.5666666666666668,
   'novelty': 0.30000000000000004,
   'relevance': 0.75,
   'quality': 0.65,
   'qualitative_feedback': "I appreciate how the story reflects the struggles many of us face in today's hyper-digital world. Chad's journey resonates with me because it highlights the search for authenticity amidst all the noise. His experience of feeling lost and then finding clarity through introspection is something I can relate to, especially considering how easy it is to get caught up in the expectations of social media. Plus, the commentary on the absurdities of online culture really hits home, as I often find myself questioning the value of all t

SyntaxError: unterminated string literal (detected at line 1) (1845220801.py, line 1)